In [ ]:
# -*- coding: utf-8 -*-
"""
不动产 GDB 数据处理（分类合并 + 重叠检查 + 分区县统计）
运行环境：ArcGIS Pro 自带 Python（arcpy / pandas / openpyxl）
可直接粘贴到 ArcGIS Pro Notebook 的单元格中运行，最后调用 main() 即可。

流程：
  1. 盘点源库（动态识别任意数量区县 GDB）
  2. 按 BDCDYH[12:14] 分类，字段并集合并到 不动产分类统计.gdb
  3. 每个分类自身重叠 + 分类两两重叠（arcpy.analysis.Intersect），写入 不动产重叠区域.gdb
  4. 计算椭球面积 OverlapArea_Sqm（AREA_GEODESIC, 平方米）
  5. 分区县聚合（跨区双向计入），导出多 Sheet Excel（Sheet 名 = 重叠图层名）
  6. 输出盘点 CSV、处理报告 CSV、处理日志 TXT

源 GDB 全程只读：只做 Describe / GetCount / ExportFeatures（读取），所有写入都在结果目录。
"""

import csv
import itertools
import logging
import re
import time
import traceback
from collections import OrderedDict, defaultdict
from pathlib import Path

import arcpy

# ============================================================
# 0. 配置区
# ============================================================
SOURCE_DIR = Path(r"D:\Pro Projects\2026不动产数据处理\第一批")
RESULT_DIR = Path(r"D:\Pro Projects\2026不动产数据处理\第一批处理结果")

KEYWORD = "不动产登记数据"                       # 行政区名称截取关键字
SOURCE_LAYERS = ["SHYQZD_TX", "SUYQZD_TX"]       # 使用权宗地 / 所有权宗地
CLASSES = ["JA", "JC", "JB", "GB", "GE", "GL", "GD", "GN"]
UNCLASSIFIED = "未分类"
BDCDYH_FIELD = "BDCDYH"
SLICE_START, SLICE_END = 12, 14                  # BDCDYH[12:14] 即第 13-14 位

SRC_XZQ_FIELD = "SourceXZQ"
AREA_FIELD = "OverlapArea_Sqm"

CLASS_GDB_NAME = "不动产分类统计.gdb"
OVERLAP_GDB_NAME = "不动产重叠区域.gdb"
SCRATCH_GDB_NAME = "scratch.gdb"
EXCEL_NAME = "分区县重叠图斑面积和数量统计.xlsx"

# 测试用：只处理前 N 个 GDB（None = 全部）；也可用 ONLY_GDB_KEYWORDS 指定名称包含的关键字
#LIMIT_GDB_COUNT = None
LIMIT_GDB_COUNT = 2
ONLY_GDB_KEYWORDS = []          # 例如 ["锦江区", "青羊区"]

CLEAN_START = True              # True: 每次运行先删除结果目录中三个同名 GDB 重新生成

SYSTEM_FIELDS = {"objectid", "oid", "fid", "shape", "shape_length", "shape_area",
                 "shape.area", "shape.len", "shape.starea()", "shape.stlength()"}
TMP_CLS_FIELD = "CLS_TMP"

# ============================================================
# 全局：日志与报告
# ============================================================
LOGGER = logging.getLogger("bdc")
REPORT_ROWS = []   # 处理报告.csv 的行


def setup_logging(result_dir):
    result_dir.mkdir(parents=True, exist_ok=True)
    LOGGER.setLevel(logging.INFO)
    for h in list(LOGGER.handlers):
        LOGGER.removeHandler(h)
        h.close()
    fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s", "%Y-%m-%d %H:%M:%S")
    fh = logging.FileHandler(result_dir / "处理日志.txt", mode="w", encoding="utf-8")
    fh.setFormatter(fmt)
    sh = logging.StreamHandler()
    sh.setFormatter(fmt)
    LOGGER.addHandler(fh)
    LOGGER.addHandler(sh)


def add_report(step, obj, status, count="", message=""):
    REPORT_ROWS.append({"步骤": step, "对象": obj, "状态": status, "数量": count, "说明": message})


def write_csv(path, rows, fieldnames):
    with open(path, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for r in rows:
            w.writerow(r)


def safe_delete(path):
    try:
        if path and arcpy.Exists(str(path)):
            arcpy.management.Delete(str(path))
    except Exception as e:  # 释放锁失败不影响主流程
        LOGGER.warning("删除临时对象失败 %s: %s", path, e)


def setup_env(result_dir):
    arcpy.env.overwriteOutput = True
    arcpy.env.scratchWorkspace = str(result_dir / SCRATCH_GDB_NAME)
    arcpy.env.parallelProcessingFactor = "80%"
    arcpy.env.addOutputsToMap = False


# ============================================================
# 1. 行政区名称解析
# ============================================================
def parse_xzq_name(gdb_name):
    """'510104锦江区不动产登记数据_20260720.gdb' -> '510104锦江区'；解析失败返回 None"""
    stem = gdb_name[:-4] if gdb_name.lower().endswith(".gdb") else gdb_name
    idx = stem.find(KEYWORD)
    if idx <= 0:
        return None
    return stem[:idx]


def split_code_name(xzq):
    m = re.match(r"^(\d+)(.*)$", xzq)
    return (m.group(1), m.group(2)) if m else ("", xzq)


# ============================================================
# 2. 源库盘点（只读）
# ============================================================
def _find_layers_in_gdb(gdb_path):
    """返回 (目标图层 {大写名: 路径}, 库内全部要素类名列表)"""
    wanted = {n.upper() for n in SOURCE_LAYERS}
    found, all_names = {}, []
    for dirpath, _dirs, files in arcpy.da.Walk(str(gdb_path), datatype="FeatureClass"):
        for fn in files:
            all_names.append(fn)
            if fn.upper() in wanted:
                found[fn.upper()] = str(Path(dirpath) / fn)
    return found, all_names


def inventory_source_gdbs(source_dir):
    """
    动态盘点源目录下全部 *.gdb，不限数量。
    返回 districts: list[dict(gdb, xzq, code, name, layers{LAYER: path}, valid)]
    同时写出 盘点_源库对象清单.csv / 盘点_行政区清单.csv
    """
    if not source_dir.exists():
        raise FileNotFoundError(f"源目录不存在: {source_dir}")

    gdbs = sorted(p for p in source_dir.iterdir() if p.is_dir() and p.suffix.lower() == ".gdb")
    if ONLY_GDB_KEYWORDS:
        gdbs = [p for p in gdbs if any(k in p.name for k in ONLY_GDB_KEYWORDS)]
    if LIMIT_GDB_COUNT:
        gdbs = gdbs[:LIMIT_GDB_COUNT]
    if not gdbs:
        raise FileNotFoundError(f"源目录下未发现任何 .gdb: {source_dir}")

    districts, obj_rows, xzq_rows = [], [], []
    for gdb in gdbs:
        xzq = parse_xzq_name(gdb.name)
        info = {"gdb": gdb, "xzq": xzq, "layers": {}, "valid": False}
        if xzq is None:
            LOGGER.warning("无法从 GDB 名称解析行政区（缺少“%s”）: %s", KEYWORD, gdb.name)
            add_report("盘点", gdb.name, "跳过", "", "名称不含关键字，无法解析行政区")
            xzq_rows.append({"行政区代码": "", "行政区名称": "", "GDB名称": gdb.name, "有效": "否",
                             "SHYQZD_TX存在": "", "SUYQZD_TX存在": "", "要素总数": 0, "备注": "名称解析失败"})
            continue
        code, name = split_code_name(xzq)
        info.update(code=code, name=name)

        try:
            found, all_names = _find_layers_in_gdb(gdb)
        except Exception as e:
            LOGGER.error("读取 GDB 失败 %s: %s", gdb.name, e)
            add_report("盘点", gdb.name, "失败", "", str(e))
            continue

        total = 0
        for lname in SOURCE_LAYERS:
            path = found.get(lname.upper())
            if not path:
                obj_rows.append({"GDB": gdb.name, "行政区": xzq, "图层": lname, "是否存在": "否",
                                 "要素数": 0, "几何类型": "", "坐标系": "", "字段数": 0, "字段清单": ""})
                continue
            try:
                desc = arcpy.Describe(path)
                cnt = int(arcpy.management.GetCount(path)[0])
                fields = [f.name for f in desc.fields]
                sr = desc.spatialReference
                obj_rows.append({"GDB": gdb.name, "行政区": xzq, "图层": lname, "是否存在": "是",
                                 "要素数": cnt, "几何类型": desc.shapeType,
                                 "坐标系": f"{sr.name}({sr.factoryCode})",
                                 "字段数": len(fields), "字段清单": ",".join(fields)})
                if desc.shapeType != "Polygon":
                    LOGGER.warning("%s/%s 不是面图层(%s)，跳过", gdb.name, lname, desc.shapeType)
                    continue
                info["layers"][lname.upper()] = path
                total += cnt
            except Exception as e:
                LOGGER.error("盘点图层失败 %s/%s: %s", gdb.name, lname, e)
                add_report("盘点", f"{gdb.name}/{lname}", "失败", "", str(e))

        info["valid"] = bool(info["layers"])
        if not info["valid"]:
            LOGGER.warning("GDB 内未找到有效目标图层: %s；库内要素类: %s", gdb.name, all_names)
            add_report("盘点", gdb.name, "无效", "", f"未找到 SHYQZD_TX/SUYQZD_TX；库内要素类={all_names}")
        xzq_rows.append({"行政区代码": code, "行政区名称": name, "GDB名称": gdb.name,
                         "有效": "是" if info["valid"] else "否",
                         "SHYQZD_TX存在": "是" if "SHYQZD_TX" in info["layers"] else "否",
                         "SUYQZD_TX存在": "是" if "SUYQZD_TX" in info["layers"] else "否",
                         "要素总数": total, "备注": ""})
        districts.append(info)

    RESULT_DIR.mkdir(parents=True, exist_ok=True)
    write_csv(RESULT_DIR / "盘点_源库对象清单.csv", obj_rows,
              ["GDB", "行政区", "图层", "是否存在", "要素数", "几何类型", "坐标系", "字段数", "字段清单"])
    write_csv(RESULT_DIR / "盘点_行政区清单.csv", xzq_rows,
              ["行政区代码", "行政区名称", "GDB名称", "有效", "SHYQZD_TX存在", "SUYQZD_TX存在", "要素总数", "备注"])

    valid = [d for d in districts if d["valid"]]
    LOGGER.info("发现 GDB %d 个，有效区县 %d 个", len(gdbs), len(valid))
    add_report("盘点", "有效区县数", "完成", len(valid), "、".join(d["xzq"] for d in valid))
    if not valid:
        raise RuntimeError("没有任何有效区县 GDB，终止")
    return valid


# ============================================================
# 3. 初始化目标库
# ============================================================
def init_target_gdbs(result_dir):
    result_dir.mkdir(parents=True, exist_ok=True)
    paths = {}
    for key, name in (("class", CLASS_GDB_NAME), ("overlap", OVERLAP_GDB_NAME), ("scratch", SCRATCH_GDB_NAME)):
        p = result_dir / name
        # 安全保护：只允许操作结果目录内的 GDB
        assert result_dir in p.parents, "目标路径必须位于结果目录内"
        if p.exists() and CLEAN_START:
            arcpy.management.Delete(str(p))
        if not arcpy.Exists(str(p)):
            arcpy.management.CreateFileGDB(str(result_dir), name)
        paths[key] = p
    LOGGER.info("目标库初始化完成")
    return paths


# ============================================================
# 4. 提取、分类、字段并集合并
# ============================================================
_TYPE_MAP = {"String": "TEXT", "Integer": "LONG", "SmallInteger": "SHORT", "Double": "DOUBLE",
             "Single": "FLOAT", "Date": "DATE", "Guid": "GUID", "Blob": "BLOB",
             "BigInteger": "BIGINTEGER", "DateOnly": "DATEONLY", "TimeOnly": "TIMEONLY",
             "TimestampOffset": "TIMESTAMPOFFSET"}


def collect_field_union(layer_paths):
    """收集所有源图层字段并集（不含系统字段与 SourceXZQ）"""
    union = OrderedDict()
    for p in layer_paths:
        for f in arcpy.ListFields(p):
            if f.type in ("OID", "Geometry") or f.name.lower() in SYSTEM_FIELDS:
                continue
            if f.name.upper() == SRC_XZQ_FIELD.upper() or f.name.upper() == TMP_CLS_FIELD:
                continue
            key = f.name.upper()
            if f.type not in _TYPE_MAP:
                LOGGER.warning("字段类型不支持，忽略: %s (%s)", f.name, f.type)
                continue
            if key not in union:
                union[key] = {"name": f.name, "type": f.type, "length": f.length,
                              "precision": f.precision, "scale": f.scale, "alias": f.aliasName}
            else:
                u = union[key]
                if u["type"] != f.type:
                    LOGGER.warning("字段 %s 在不同源库类型不一致(%s vs %s)，统一为文本(255)",
                                   f.name, u["type"], f.type)
                    u.update(type="String", length=255)
                elif f.type == "String":
                    u["length"] = max(u["length"], f.length)
    return union


def _create_class_fc(gdb, name, sr, union):
    fc = str(Path(gdb) / name)
    arcpy.management.CreateFeatureclass(str(gdb), name, "POLYGON", spatial_reference=sr)
    for u in union.values():
        kw = {"field_alias": u["alias"]}
        ftype = _TYPE_MAP[u["type"]]
        if ftype == "TEXT":
            kw["field_length"] = max(int(u["length"] or 255), 1)
        elif ftype in ("DOUBLE", "FLOAT") and u["precision"]:
            kw["field_precision"], kw["field_scale"] = u["precision"], u["scale"]
        arcpy.management.AddField(fc, u["name"], ftype, **kw)
    arcpy.management.AddField(fc, SRC_XZQ_FIELD, "TEXT", field_length=50)
    return fc


def _classify(value):
    if value is None:
        return UNCLASSIFIED
    s = str(value).strip()
    if len(s) < SLICE_END:
        return UNCLASSIFIED
    code = s[SLICE_START:SLICE_END]
    return code if code in CLASSES else UNCLASSIFIED


def merge_and_classify_layers(districts, class_gdb, scratch_gdb):
    """
    返回 (class_fcs {分类名: 路径}, counts {分类名: 要素数}, source_total)
    """
    all_paths = [p for d in districts for p in d["layers"].values()]
    union = collect_field_union(all_paths)
    sr_first = arcpy.Describe(all_paths[0]).spatialReference
    for p in all_paths:
        sr = arcpy.Describe(p).spatialReference
        if sr.factoryCode != sr_first.factoryCode or sr.name != sr_first.name:
            LOGGER.warning("检测到坐标系不一致: %s (%s) 与基准 %s；Append 将自动投影，请确认地理变换",
                           p, sr.name, sr_first.name)
    LOGGER.info("字段并集 %d 个；目标坐标系: %s", len(union), sr_first.name)

    class_fcs = {c: _create_class_fc(class_gdb, c, sr_first, union) for c in CLASSES + [UNCLASSIFIED]}
    counts = defaultdict(int)
    source_total = 0

    for di, d in enumerate(districts):
        for lname, path in d["layers"].items():
            tag = f"{d['xzq']}/{lname}"
            tmp = str(Path(scratch_gdb) / f"src_{di}_{lname}")
            try:
                src_cnt = int(arcpy.management.GetCount(path)[0])
                source_total += src_cnt
                if src_cnt == 0:
                    add_report("分类合并", tag, "空图层", 0, "")
                    continue
                # 读取源数据到 scratch（源库只读）
                arcpy.conversion.ExportFeatures(path, tmp)
                existing = {f.name.upper(): f.name for f in arcpy.ListFields(tmp)}
                if SRC_XZQ_FIELD.upper() not in existing:
                    arcpy.management.AddField(tmp, SRC_XZQ_FIELD, "TEXT", field_length=50)
                arcpy.management.AddField(tmp, TMP_CLS_FIELD, "TEXT", field_length=20)
                bdc = existing.get(BDCDYH_FIELD.upper())
                if not bdc:
                    LOGGER.warning("%s 缺少 %s 字段，全部归入 %s", tag, BDCDYH_FIELD, UNCLASSIFIED)

                local = defaultdict(int)
                cur_fields = [TMP_CLS_FIELD, SRC_XZQ_FIELD] + ([bdc] if bdc else [])
                with arcpy.da.UpdateCursor(tmp, cur_fields) as cur:
                    for row in cur:
                        cls = _classify(row[2]) if bdc else UNCLASSIFIED
                        row[0], row[1] = cls, d["xzq"]
                        cur.updateRow(row)
                        local[cls] += 1

                delim = arcpy.AddFieldDelimiters(tmp, TMP_CLS_FIELD)
                for cls, n in local.items():
                    lyr = f"lyr_{di}_{lname}_{cls}"
                    try:
                        arcpy.management.MakeFeatureLayer(tmp, lyr, f"{delim} = '{cls}'")
                        arcpy.management.Append([lyr], class_fcs[cls], "NO_TEST")
                        counts[cls] += n
                    finally:
                        safe_delete(lyr)
                add_report("分类合并", tag, "完成", src_cnt,
                           "; ".join(f"{k}={v}" for k, v in sorted(local.items())))
                LOGGER.info("合并完成 %s: %d 条 -> %s", tag, src_cnt, dict(local))
            except Exception as e:
                LOGGER.error("合并失败 %s: %s\n%s", tag, e, traceback.format_exc())
                add_report("分类合并", tag, "失败", "", str(e))
            finally:
                safe_delete(tmp)

    # 完整性校验
    merged_total = 0
    for c, fc in list(class_fcs.items()):
        n = int(arcpy.management.GetCount(fc)[0])
        merged_total += n
        if n != counts.get(c, 0):
            LOGGER.warning("分类 %s 计数不一致：累计=%d，实际=%d", c, counts.get(c, 0), n)
        counts[c] = n
        add_report("分类结果", c, "完成", n, "")
    if counts[UNCLASSIFIED] == 0:        # 无未分类数据则不保留该图层
        safe_delete(class_fcs.pop(UNCLASSIFIED))
        counts.pop(UNCLASSIFIED, None)
    ok = merged_total == source_total
    LOGGER.info("要素总量校验：源=%d，合并=%d，%s", source_total, merged_total, "一致" if ok else "不一致！")
    add_report("完整性校验", "源 vs 合并", "通过" if ok else "不通过", merged_total, f"源总量={source_total}")
    return class_fcs, dict(counts), source_total


# ============================================================
# 5. 重叠检查（Intersect）
# ============================================================
def _build_slim_pair(cls_fc, cls, scratch_gdb):
    """
    为 Intersect 生成两份轻量副本，字段名固定：
      S1_<cls>: OID_1, XZQ_1, BDCDYH_1      S2_<cls>: OID_2, XZQ_2, BDCDYH_2
    这样相交结果字段名确定，不依赖 ArcGIS 的自动重命名规则。
    """
    desc = arcpy.Describe(cls_fc)
    sr = desc.spatialReference
    names = {f.name.upper(): f.name for f in arcpy.ListFields(cls_fc)}
    bdc = names.get(BDCDYH_FIELD.upper())
    xzq = names[SRC_XZQ_FIELD.upper()]

    s1 = str(Path(scratch_gdb) / f"S1_{cls}")
    s2 = str(Path(scratch_gdb) / f"S2_{cls}")
    safe_delete(s1)
    safe_delete(s2)

    # 1) 只保留 SourceXZQ / BDCDYH 两个属性字段，导出为副本1（无游标，避免事务冲突）
    keep = {xzq.upper()} | ({bdc.upper()} if bdc else set())
    fms = arcpy.FieldMappings()
    fms.addTable(cls_fc)
    for i in reversed(range(fms.fieldCount)):
        if fms.getFieldMap(i).outputField.name.upper() not in keep:
            fms.removeFieldMap(i)
    arcpy.conversion.ExportFeatures(cls_fc, s1, field_mapping=fms)

    # 2) 统一字段名：XZQ_1 / BDCDYH_1 / OID_1
    arcpy.management.AlterField(s1, xzq, "XZQ_1", "XZQ_1")
    if bdc:
        arcpy.management.AlterField(s1, bdc, "BDCDYH_1", "BDCDYH_1")
    else:
        arcpy.management.AddField(s1, "BDCDYH_1", "TEXT", field_length=50)
    arcpy.management.AddField(s1, "OID_1", "LONG")
    oid_name = arcpy.Describe(s1).OIDFieldName
    arcpy.management.CalculateField(s1, "OID_1", f"!{oid_name}!", "PYTHON3")

    # 3) 副本2由副本1复制而来，字段改名为 _2，OID_2 的值与 OID_1 完全一致
    arcpy.management.CopyFeatures(s1, s2)
    arcpy.management.AlterField(s2, "XZQ_1", "XZQ_2", "XZQ_2")
    arcpy.management.AlterField(s2, "BDCDYH_1", "BDCDYH_2", "BDCDYH_2")
    arcpy.management.AlterField(s2, "OID_1", "OID_2", "OID_2")

    outs = [s1, s2]
    for fc in outs:
        try:
            arcpy.management.RepairGeometry(fc)
        except Exception as e:
            LOGGER.warning("修复几何失败 %s: %s", fc, e)
    return outs



def execute_overlap_intersect(class_fcs, counts, overlap_gdb, scratch_gdb):
    """返回成功生成的重叠图层名列表（与 Sheet 一一对应）"""
    created = []
    classes = [c for c in list(CLASSES) + [UNCLASSIFIED] if c in class_fcs and counts.get(c, 0) > 0]
    skipped = [c for c in class_fcs if c not in classes]
    if skipped:
        LOGGER.info("以下分类无要素，不参与重叠检查: %s", skipped)

    slims = {}
    for c in classes:
        try:
            slims[c] = _build_slim_pair(class_fcs[c], c, scratch_gdb)
        except Exception as e:
            LOGGER.error("构建 %s 副本失败: %s", c, e)
            add_report("重叠检查", c, "失败", "", f"构建副本失败: {e}")
    classes = [c for c in classes if c in slims]

    # 任务清单：先自身，再两两
    tasks = [(c, c, f"{c}_自身重叠") for c in classes]
    tasks += [(a, b, f"{a}_{b}_重叠") for a, b in itertools.combinations(classes, 2)]

    for a, b, out_name in tasks:
        t0 = time.time()
        out_fc = str(Path(overlap_gdb) / out_name)
        tmp_out = str(Path(scratch_gdb) / f"tmp_{out_name}")
        try:
            in1, in2 = slims[a][0], slims[b][1]
            if a == b:
                # 自身：相交后仅保留 OID_1 < OID_2（排除与自身相交，并去除 AB/BA 重复）
                arcpy.analysis.Intersect([in1, in2], tmp_out, "ALL", None, "INPUT")
                arcpy.analysis.Select(tmp_out, out_fc, "OID_1 < OID_2")
            else:
                arcpy.analysis.Intersect([in1, in2], out_fc, "ALL", None, "INPUT")
            n = int(arcpy.management.GetCount(out_fc)[0])
            created.append(out_name)
            LOGGER.info("重叠 %s: %d 个图斑 (%.1fs)", out_name, n, time.time() - t0)
            add_report("重叠检查", out_name, "完成", n, f"耗时{time.time() - t0:.1f}s")
        except Exception as e:
            LOGGER.error("重叠检查失败 %s: %s\n%s", out_name, e, traceback.format_exc())
            add_report("重叠检查", out_name, "失败", "", str(e))
            safe_delete(out_fc)
        finally:
            safe_delete(tmp_out)

    for pair in slims.values():
        for fc in pair:
            safe_delete(fc)
    return created


# ============================================================
# 6. 椭球面积
# ============================================================
def calculate_geodesic_areas(overlap_gdb, layer_names):
    for name in layer_names:
        fc = str(Path(overlap_gdb) / name)
        try:
            if AREA_FIELD.upper() not in [f.name.upper() for f in arcpy.ListFields(fc)]:
                arcpy.management.AddField(fc, AREA_FIELD, "DOUBLE")
            n = int(arcpy.management.GetCount(fc)[0])
            if n > 0:
                arcpy.management.CalculateGeometryAttributes(
                    fc, [[AREA_FIELD, "AREA_GEODESIC"]], area_unit="SQUARE_METERS")
            add_report("椭球面积", name, "完成", n, "AREA_GEODESIC / SQUARE_METERS")
        except Exception as e:
            LOGGER.error("椭球面积计算失败 %s: %s", name, e)
            add_report("椭球面积", name, "失败", "", str(e))


# ============================================================
# 7. 分区县聚合与 Excel 导出
# ============================================================
def _aggregate_overlap(fc):
    """
    同区重叠：该区 +1；跨区重叠：双方各 +1（用 set 去重天然实现）。
    返回 {SourceXZQ: [数量, 面积]}
    """
    stat = defaultdict(lambda: [0, 0.0])
    with arcpy.da.SearchCursor(fc, ["XZQ_1", "XZQ_2", AREA_FIELD]) as cur:
        for x1, x2, area in cur:
            area = float(area or 0.0)
            for key in {x1 or "未知", x2 or "未知"}:
                stat[key][0] += 1
                stat[key][1] += area
    return stat


def export_overlap_excel_report(districts, overlap_gdb, result_dir):
    import pandas as pd

    # 以重叠库中实际存在的图层为准，保证 Sheet 与图层一一对应
    arcpy.env.workspace = str(overlap_gdb)
    layer_names = sorted(arcpy.ListFeatureClasses() or [],
                         key=lambda n: (0 if n.endswith("_自身重叠") else 1, n))
    arcpy.env.workspace = None

    xlsx = result_dir / EXCEL_NAME
    cols = ["行政区代码", "行政区名称", "重叠图斑数(个)", "重叠图斑面积(平方米)"]
    if not layer_names:
        LOGGER.warning("重叠库内没有图层，不生成 Excel")
        add_report("Excel", EXCEL_NAME, "跳过", 0, "无重叠图层")
        return layer_names

    with pd.ExcelWriter(xlsx, engine="openpyxl") as writer:
        for name in layer_names:
            assert len(name) <= 31, f"Sheet 名超过 31 字符: {name}"
            fc = str(Path(overlap_gdb) / name)
            try:
                stat = _aggregate_overlap(fc)
            except Exception as e:
                LOGGER.error("统计失败 %s: %s", name, e)
                stat = {}
                add_report("Excel", name, "失败", "", str(e))
            rows, used = [], set()
            for d in districts:
                n, a = stat.get(d["xzq"], [0, 0.0])
                rows.append([d["code"], d["name"], int(n), round(a, 2)])
                used.add(d["xzq"])
            for key, (n, a) in stat.items():       # 不在区县清单内的来源（正常不会出现）
                if key not in used:
                    code, nm = split_code_name(key)
                    rows.append([code, nm, int(n), round(a, 2)])
            df = pd.DataFrame(rows, columns=cols)
            df["重叠图斑面积(平方米)"] = df["重叠图斑面积(平方米)"].map(lambda v: float(f"{v:.2f}"))
            df.to_excel(writer, sheet_name=name, index=False)
            ws = writer.sheets[name]
            ws.column_dimensions["A"].width = 14
            ws.column_dimensions["B"].width = 16
            ws.column_dimensions["C"].width = 16
            ws.column_dimensions["D"].width = 24
            for r in range(2, ws.max_row + 1):
                ws.cell(r, 4).number_format = "#,##0.00"
            add_report("Excel", name, "完成", int(df["重叠图斑数(个)"].sum()), "")

    # Sheet 与图层一致性校验
    from openpyxl import load_workbook
    sheets = load_workbook(xlsx, read_only=True).sheetnames
    same = (sheets == layer_names) or (set(sheets) == set(layer_names) and len(sheets) == len(layer_names))
    LOGGER.info("Excel 已生成：%d 个 Sheet / %d 个重叠图层，%s",
                len(sheets), len(layer_names), "一致" if same else "不一致！")
    add_report("Excel校验", "Sheet vs 图层", "通过" if same else "不通过", len(sheets),
               f"图层数={len(layer_names)}")
    return layer_names


# ============================================================
# 8. 报告
# ============================================================
def write_summary_report(result_dir):
    write_csv(result_dir / "处理报告.csv", REPORT_ROWS, ["步骤", "对象", "状态", "数量", "说明"])
    LOGGER.info("处理报告已写出：%s", result_dir / "处理报告.csv")


# ============================================================
# 9. 主入口
# ============================================================
def main():
    t_start = time.time()
    REPORT_ROWS.clear()
    setup_logging(RESULT_DIR)
    LOGGER.info("===== 不动产 GDB 数据处理开始 =====")
    try:
        # 先建 scratch，再设环境
        RESULT_DIR.mkdir(parents=True, exist_ok=True)
        targets = init_target_gdbs(RESULT_DIR)
        setup_env(RESULT_DIR)

        districts = inventory_source_gdbs(SOURCE_DIR)

        class_fcs, counts, src_total = merge_and_classify_layers(
            districts, targets["class"], targets["scratch"])

        created = execute_overlap_intersect(class_fcs, counts, targets["overlap"], targets["scratch"])
        calculate_geodesic_areas(targets["overlap"], created)
        export_overlap_excel_report(districts, targets["overlap"], RESULT_DIR)

        LOGGER.info("分类统计：%s", counts)
        add_report("总览", "分类要素数", "完成", sum(counts.values()), str(counts))
    except Exception as e:
        LOGGER.error("主流程异常终止: %s\n%s", e, traceback.format_exc())
        add_report("主流程", "main", "异常终止", "", str(e))
    finally:
        # 清理 scratch 中残留的临时对象
        try:
            arcpy.env.workspace = str(RESULT_DIR / SCRATCH_GDB_NAME)
            for fc in (arcpy.ListFeatureClasses() or []):
                safe_delete(Path(arcpy.env.workspace) / fc)
            arcpy.env.workspace = None
        except Exception:
            pass
        LOGGER.info("===== 处理结束，总耗时 %.1f 分钟 =====", (time.time() - t_start) / 60)
        write_summary_report(RESULT_DIR)
        for h in LOGGER.handlers:
            h.flush()


# 在 ArcGIS Pro Notebook 中直接运行：
main()

2026-10-13 11:57:19 [INFO] ===== 不动产 GDB 数据处理开始 =====
2026-10-13 11:57:20 [INFO] 目标库初始化完成
2026-10-13 11:57:20 [INFO] 发现 GDB 2 个，有效区县 2 个
2026-10-13 11:57:21 [INFO] 字段并集 179 个；目标坐标系: cd
